#  Libraries imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import (mean_squared_error, r2_score, classification_report, 
                              confusion_matrix, accuracy_score, f1_score, 
                              roc_curve, roc_auc_score, precision_score, recall_score)

sns.set_theme(style="whitegrid")

#   Load data

In [ ]:
df = pd.read_csv('data/cleaned_data.csv')
df.shape

#   Feature Engineering & Encoding

##  nulls fill 

In [ ]:
df['Traffic Control Presence'] = df['Traffic Control Presence'].fillna('None')
df['Driver License Status'] = df['Driver License Status'].fillna('Unknown')

##  Seperate Target Column

In [ ]:
y_reg = df['Number of Casualties']
y_clf = (df['Accident Severity'] == 'Fatal').astype(int)

X = df.drop(columns=['Number of Casualties', 'Accident Severity'])

print("y_reg sample:", y_reg.head().tolist())
print("y_clf distribution:\n", y_clf.value_counts())

##  Identify Categorical Columns

In [ ]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
print(categorical_cols)

## Check Cardinality of Categorical Columns

In [ ]:
for col in categorical_cols:
    print(col, '->', X[col].nunique(), 'unique values')

In [ ]:
df['Time of Day'].head(10)

##  Convert "Time of Day" into Time Buckets

In [ ]:
def get_time_bucket(time_str):
    hour = int(time_str.split(':')[0])
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

X['Time Bucket'] = X['Time of Day'].apply(get_time_bucket)
X['Time Bucket'].value_counts()

## Drop Redundant/High-Cardinality Columns

In [ ]:
X = X.drop(columns=['City Name', 'Time of Day'])
X.shape

## One-Hot Encode Categorical Columns

In [ ]:
categorical_cols_final = X.select_dtypes(include='object').columns.tolist()
X_encoded = pd.get_dummies(X, columns=categorical_cols_final, drop_first=True)
X_encoded.shape

## Train/Test Split
#### Splitting data into 80% training and 20% testing, with stratification to preserve severity class balance.

In [ ]:
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X_encoded, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
X_train.shape, X_test.shape

## Scale Numeric Features
#### Logistic Regression is sensitive to feature scale, so we standardize all features before training.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Regression Model — Linear Regression
#### Predicting Number of Casualties using Linear Regression.

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_reg_train)
y_pred_reg = lin_reg.predict(X_test_scaled)

mse = mean_squared_error(y_reg_test, y_pred_reg)
r2 = r2_score(y_reg_test, y_pred_reg)

print(f"MSE: {mse:.4f}")
print(f"R²: {r2:.4f}")

## Linear Regression — Feature Coefficients

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X_encoded.columns,
    'Coefficient': lin_reg.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print(coef_df.head(3))

## Regression Model — Ridge Regression

In [ ]:
ridge_reg = Ridge(alpha=1.0)
ridge_reg.fit(X_train_scaled, y_reg_train)
y_pred_ridge = ridge_reg.predict(X_test_scaled)

mse_ridge = mean_squared_error(y_reg_test, y_pred_ridge)
r2_ridge = r2_score(y_reg_test, y_pred_ridge)

print(f"Ridge MSE: {mse_ridge:.4f}")
print(f"Ridge R²: {r2_ridge:.4f}")

reg_comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression'],
    'MSE': [mse, mse_ridge],
    'R2': [r2, r2_ridge]
})
print(reg_comparison)

## Classification — Class Imbalance Check

In [ ]:
print(y_clf_train.value_counts())
print(f"\nMinority class %: {y_clf_train.value_counts(normalize=True).min()*100:.2f}%")

## Classification Model — Logistic Regression (with class_weight='balanced')
#### Since the minority class (Fatal) is 32.83% of the training data — below the 35% threshold — class_weight='balanced' is used to address the imbalance, chosen over SMOTE for simplicity and because it requires no additional resampling library.

In [ ]:
clf_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
clf_model.fit(X_train_scaled, y_clf_train)

y_pred_clf = clf_model.predict(X_test_scaled)
y_proba_clf = clf_model.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_clf_test, y_pred_clf))
print(classification_report(y_clf_test, y_pred_clf, target_names=['Not-Fatal', 'Fatal']))

## Confusion Matrix — Logistic Regression

In [ ]:
cm = confusion_matrix(y_clf_test, y_pred_clf)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='rocket_r', 
            xticklabels=['Not-Fatal', 'Fatal'], yticklabels=['Not-Fatal', 'Fatal'])
plt.title('Confusion Matrix — Logistic Regression', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('plots/confusion_matrix_logistic.png', dpi=150, bbox_inches='tight')
plt.show()

## ROC Curve & AUC

In [ ]:
fpr, tpr, thresholds = roc_curve(y_clf_test, y_proba_clf)
auc_score = roc_auc_score(y_clf_test, y_proba_clf)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='crimson', linewidth=2, label=f'ROC Curve (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Logistic Regression', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('plots/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC: {auc_score:.4f}")

## Decision Threshold Sensitivity

In [ ]:
thresholds_to_test = [0.30, 0.40, 0.50, 0.60, 0.70]
threshold_results = []

for t in thresholds_to_test:
    y_pred_t = (y_proba_clf >= t).astype(int)
    p = precision_score(y_clf_test, y_pred_t, zero_division=0)
    r = recall_score(y_clf_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_clf_test, y_pred_t, zero_division=0)
    threshold_results.append({'Threshold': t, 'Precision': p, 'Recall': r, 'F1': f1})

threshold_df = pd.DataFrame(threshold_results)
print(threshold_df)

## Regularization Experiment — C=0.01 vs C=1.0

In [ ]:
clf_strong_reg = LogisticRegression(max_iter=1000, random_state=42, 
                                      class_weight='balanced', C=0.01)
clf_strong_reg.fit(X_train_scaled, y_clf_train)
y_pred_strong = clf_strong_reg.predict(X_test_scaled)
y_proba_strong = clf_strong_reg.predict_proba(X_test_scaled)[:, 1]

precision_baseline = precision_score(y_clf_test, y_pred_clf)
recall_baseline = recall_score(y_clf_test, y_pred_clf)
auc_baseline = roc_auc_score(y_clf_test, y_proba_clf)

precision_strong = precision_score(y_clf_test, y_pred_strong)
recall_strong = recall_score(y_clf_test, y_pred_strong)
auc_strong = roc_auc_score(y_clf_test, y_proba_strong)

reg_comparison = pd.DataFrame({
    'Model': ['C=1.0 (baseline)', 'C=0.01 (strong regularization)'],
    'Precision': [precision_baseline, precision_strong],
    'Recall': [recall_baseline, recall_strong],
    'AUC': [auc_baseline, auc_strong]
})
print(reg_comparison)

## Bootstrap Confidence Interval — AUC Difference (C=1.0 vs C=0.01)

In [ ]:
np.random.seed(42)
n_bootstrap = 500
auc_diffs = []

y_clf_test_arr = y_clf_test.values
n = len(y_clf_test_arr)

for i in range(n_bootstrap):
    idx = np.random.choice(n, size=n, replace=True)
    y_sample = y_clf_test_arr[idx]
    
    if len(np.unique(y_sample)) < 2:
        continue
    
    proba_baseline_sample = y_proba_clf[idx]
    proba_strong_sample = y_proba_strong[idx]
    
    auc_base = roc_auc_score(y_sample, proba_baseline_sample)
    auc_str = roc_auc_score(y_sample, proba_strong_sample)
    auc_diffs.append(auc_base - auc_str)

auc_diffs = np.array(auc_diffs)
mean_diff = auc_diffs.mean()
ci_lower = np.percentile(auc_diffs, 2.5)
ci_upper = np.percentile(auc_diffs, 97.5)

print(f"Mean AUC difference (C=1.0 minus C=0.01): {mean_diff:.4f}")
print(f"95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"CI excludes zero: {ci_lower > 0 or ci_upper < 0}")

## Conclusion
#### Across regression (negative R²), classification (50.8% accuracy, AUC 0.523), threshold sensitivity, and regularization experiments, every model consistently performs at or near random-chance level. The bootstrap confidence interval confirms even the small differences between regularization strengths are not statistically reliable. This is not a flaw in the modeling approach — preprocessing, encoding, scaling, and imbalance handling were all done correctly — but a property of the dataset itself, consistent with Part 1's finding that this data appears synthetically/uniformly generated rather than reflecting genuine real-world accident dynamics.